# HYPER-3, 1 — The trial, and the synthetic world it runs in

**The question someone actually asked.** *We have a new antihypertensive. We want to
know which of three daily doses to take into phase III, in adults who already have
high blood pressure or are on their way to it. We are going to watch these people
every week for six months. If one of the doses is making people worse than the care
they would have got anyway, we want to stop that arm before we find out the hard way.*

That request contains a design, an identification problem, a dose–response problem,
and a sequential-monitoring problem, and it is the spine of this six-notebook series:

| | |
|---|---|
| **1 — this notebook** | the protocol, the synthetic world, and what the data look like |
| **2** | what randomization licenses, and what it does not (`axiom.identify`, `axiom.estimands`) |
| **3** | how big, allocated how, powered for what (`axiom.design`) |
| **4** | the dose–response surface, and which doses phase III should test (`axiom.surface`) |
| **5** | the sequential design: boundaries and operating characteristics (`axiom.design.sequential`) |
| **6** | running it — the interim that stops an arm, and the final readout |

## HYPER-3

400 adults with elevated or stage-1 hypertension, randomized 2:1:1:1 to the standard
of care or to 10, 20 or 40 mg of the investigational agent daily. Randomization is
**stratified by age band** — 25–35, 36–50, 51+ — because pressure, its variability,
and drug clearance all change with age. Seated systolic pressure is measured
**weekly** for 24 weeks. Enrollment is staggered over 16 calendar weeks, so the trial
runs for 40 calendar weeks and at any interim review some units have half a year of
follow-up and some have a month.

The synthetic world lives in `hyper3.py` next to this notebook. Everything the
notebooks conclude is computed with `axiom`; `hyper3` only supplies the trial and the
house plotting style.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import hyper3 as h
from axiom.core import (
    BASES, D, Covariate, Dose, Outcome, Population, TimeWindow, Treatment, Unit, dimensionless,
)
from axiom.data import Panel, RoleMap

SEED = 20260821
trial = h.trial(seed=SEED)
print(f"{trial.n_units} units, {len(trial.visits):,} weekly measurements, "
      f"{h.TRIAL_WEEKS} calendar weeks")
print(f"arms: {', '.join(h.ARM_LABEL[a] for a in h.ARMS)}")
print(f"strata: {', '.join(h.STRATUM_LABEL[s] for s in h.STRATA)}")
trial.arm_counts()

## 1. The entities, before any data

axiom's vocabulary is the clinical one already: a `Treatment` you can intervene on, a
`Dose` with a unit, an `Outcome` you are trying to move, `Covariate`s you measured but
did not set, and a `Population` with the strata weights that define it. Declaring them
first is not ceremony — the dimensions are what stop a pressure being subtracted from
a milligram later on.

In [ ]:
# `BASES` ships four bases — time, currency, outcome, entity. A dose in milligrams needs
# a fifth, and declaring it is idempotent: the second caller gets the same dimension.
mass = BASES.declare("mass", symbol="M")
print("declared bases:", sorted(BASES.names), "-> dose dimension", mass)

patient = Unit(name="patient", dimension=D.entity, kind="individual",
               description="one enrolled adult")
sbp = Outcome(name="sbp_change", dimension=D.outcome, unit=h.OUTCOME_UNIT,
              description="change in seated systolic pressure from that unit's own baseline",
              aggregation="mean")
drug = Treatment(name="dose", dimension=D.mass, unit=h.DOSE_UNIT,
                 description="daily dose of the investigational agent")
daily = Dose(name="dose", dimension=D.mass, unit=h.DOSE_UNIT, numeraire=h.DOSE_UNIT)
baseline = Covariate(name="baseline_sbp", dimension=D.outcome, unit=h.OUTCOME_UNIT,
                     description="pre-randomization seated systolic pressure")
adherence = Covariate(name="adherence", dimension=dimensionless(),
                      description="share of prescribed tablets taken — measured, and post-randomization")
population = Population(name="hypertensive_adults_25_plus",
                        description="adults 25+ with elevated or stage-1 hypertension",
                        strata={"age_band": dict(h.STRATUM_SHARE)})
primary_window = TimeWindow(start=h.WINDOW["primary"][0], stop=h.WINDOW["primary"][1])

for entity in (sbp, drug, daily, baseline, adherence, patient):
    print(f"{entity.name:14s} {str(entity.dimension):>3s}  {entity.unit or '-':>5s}   {type(entity).__name__}")
print("\npopulation strata:", population.strata["age_band"])
print("primary window: weeks", primary_window.start, "-", primary_window.stop)

## 2. Randomization worked

The first plot of any trial is the one nobody publishes: did the randomization
actually balance the things it was supposed to balance? Baseline pressure differs by
**stratum** — that is the design — and should not differ by **arm** within a stratum.

In [ ]:
units = trial.units
fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[h.STRATUM_LABEL[s] for s in h.STRATA])
for col, stratum in enumerate(h.STRATA, start=1):
    for arm in h.ARMS:
        rows = units[(units["stratum"] == stratum) & (units["arm"] == arm)]
        fig.add_trace(
            go.Box(y=rows["baseline_sbp"], name=h.ARM_LABEL[arm], marker_color=h.ARM_COLOR[arm],
                   showlegend=col == 1, boxpoints="outliers", width=0.6),
            row=1, col=col,
        )
fig.update_layout(title="Baseline systolic pressure by arm, within stratum",
                  height=420, template="plotly_white", boxmode="group",
                  legend={"orientation": "h", "y": 1.08, "x": 0.0},
                  margin={"l": 60, "r": 30, "t": 90, "b": 40})
fig.update_yaxes(title_text=f"baseline SBP ({h.OUTCOME_UNIT})", row=1, col=1, gridcolor=h.GRID)
fig

In [ ]:
balance = h.mean_with_error(units, ["stratum", "arm"], "baseline_sbp")
table = balance.pivot(index="stratum", columns="arm", values="mean").reindex(
    index=list(h.STRATA), columns=list(h.ARMS)
)
print("mean baseline SBP by stratum and arm (mmHg)")
print(table.round(1).to_string())
spread = (table.max(axis=1) - table.min(axis=1))
print("\nlargest within-stratum gap between arms:", round(float(spread.max()), 2), h.OUTCOME_UNIT)
print("gap between strata:", round(float(table.mean(axis=1).max() - table.mean(axis=1).min()), 1), h.OUTCOME_UNIT)

The between-stratum gap is more than twice the largest within-stratum gap between
arms, which is the pattern stratified randomization is for. But the within-stratum gap
is not zero either: with 28 units on the top dose in the oldest stratum, chance alone
moves the arm means several mmHg apart. That residual imbalance is why every analysis
from here on adjusts for baseline pressure — not to remove confounding, which
randomization already handled, but to remove the variance that a 5 mmHg accident
would otherwise put straight into the contrast.

## 3. What the weekly measurements look like

Twenty-four weekly readings per unit, autocorrelated, on top of a unit-specific
baseline. One unit's series is nearly useless; four hundred of them are not.

In [ ]:
sample = units.sample(24, random_state=1)["unit"]
fig = h.figure("Twenty-four sampled units: weekly seated systolic pressure",
               "weeks since randomization", f"SBP ({h.OUTCOME_UNIT})", height=420)
for unit in sample:
    rows = trial.visits[trial.visits["unit"] == unit]
    arm = str(rows["arm"].iloc[0])
    fig.add_trace(go.Scatter(x=rows["week"], y=rows["sbp"], mode="lines",
                             line={"color": h._rgba(h.ARM_COLOR[arm], 0.55), "width": 1.4},
                             name=h.ARM_LABEL[arm], legendgroup=arm, showlegend=False,
                             hovertemplate=f"{unit} ({h.ARM_LABEL[arm]})<br>week %{{x}}: %{{y:.0f}}<extra></extra>"))
fig

## 4. The mean trajectory, pooled — and why it is a trap

Pooled over the whole trial population, the four arms separate the way a dose–response
is supposed to: more dose, more reduction. The 40 mg arm looks like the winner at
eight weeks and like a mild disappointment at twenty-four.

In [ ]:
by_week = h.mean_with_error(trial.visits, ["arm", "week"], "change")
fig = h.figure("Mean change from baseline, pooled over strata",
               "weeks since randomization", f"change in SBP ({h.OUTCOME_UNIT})", height=420)
for arm in h.ARMS:
    rows = by_week[by_week["arm"] == arm].sort_values("week")
    h.band(fig, rows["week"], rows["mean"] - 2 * rows["se"], rows["mean"] + 2 * rows["se"],
           h.ARM_COLOR[arm])
    fig.add_trace(go.Scatter(x=rows["week"], y=rows["mean"], mode="lines+markers",
                             line={"color": h.ARM_COLOR[arm], "width": 2.4},
                             marker={"size": 5}, name=h.ARM_LABEL[arm]))
fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.35)", "dash": "dot"})
fig

## 5. The same data, split by the stratum the protocol randomized on

This is the whole case study in one figure. The 40 mg arm is the best arm in the two
younger strata and, in the oldest stratum, is pushing pressure **up** — a mean nearly
5 mmHg above the standard of care and rising. Pooled, the two cancel.

In [ ]:
fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[f"{h.STRATUM_LABEL[s]}  (n={int((units['stratum'] == s).sum())})"
                                    for s in h.STRATA])
strat = h.mean_with_error(trial.visits, ["stratum", "arm", "week"], "change")
for col, stratum in enumerate(h.STRATA, start=1):
    for arm in h.ARMS:
        rows = strat[(strat["stratum"] == stratum) & (strat["arm"] == arm)].sort_values("week")
        fig.add_trace(go.Scatter(x=rows["week"], y=rows["mean"], mode="lines",
                                 line={"color": h.ARM_COLOR[arm], "width": 2.2},
                                 name=h.ARM_LABEL[arm], legendgroup=arm, showlegend=col == 1),
                      row=1, col=col)
    fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.35)", "dash": "dot"}, row=1, col=col)
    fig.update_xaxes(title_text="week", row=1, col=col, gridcolor=h.GRID)
fig.update_yaxes(title_text=f"change in SBP ({h.OUTCOME_UNIT})", row=1, col=1, gridcolor=h.GRID)
fig.update_layout(title="Mean change from baseline, by age stratum",
                  height=420, template="plotly_white",
                  legend={"orientation": "h", "y": 1.10, "x": 0.0},
                  margin={"l": 60, "r": 30, "t": 100, "b": 50})
fig

## 6. The truth behind the picture

The synthetic world has an Emax benefit that grows with dose and saturates, and a
*pressor* effect that is cubic in the dose actually taken and concentrated where
clearance is slowest. Below 20 mg the cubic term is negligible; at 40 mg in the oldest
stratum it dominates.

In [ ]:
grid = np.linspace(0.0, 48.0, 121)
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Components at week 12, oldest stratum", "Net contrast at week 12, by stratum"))
fig.add_trace(go.Scatter(x=grid, y=-h.benefit(grid, "age_51_plus", 12.0), mode="lines",
                         line={"color": "#2f7fd1", "width": 2.5}, name="benefit (Emax)"), row=1, col=1)
fig.add_trace(go.Scatter(x=grid, y=h.harm(grid, "age_51_plus", 12.0), mode="lines",
                         line={"color": "#d1483f", "width": 2.5}, name="harm (cubic)"), row=1, col=1)
fig.add_trace(go.Scatter(x=grid, y=-h.benefit(grid, "age_51_plus", 12.0) + h.harm(grid, "age_51_plus", 12.0),
                         mode="lines", line={"color": "#222", "width": 2.5, "dash": "dash"},
                         name="net"), row=1, col=1)
for stratum in h.STRATA:
    net = [h.per_protocol_contrast(float(d), stratum) for d in grid]
    fig.add_trace(go.Scatter(x=grid, y=net, mode="lines", name=h.STRATUM_LABEL[stratum],
                             line={"color": h.STRATUM_COLOR[stratum], "width": 2.5}), row=1, col=2)
for col in (1, 2):
    fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.35)", "dash": "dot"}, row=1, col=col)
    for dose in list(h.DOSE.values())[1:]:
        fig.add_vline(x=dose, line={"color": "rgba(0,0,0,0.15)"}, row=1, col=col)
    fig.update_xaxes(title_text=f"daily dose ({h.DOSE_UNIT})", row=1, col=col, gridcolor=h.GRID)
fig.update_yaxes(title_text=f"mmHg vs control", row=1, col=1, gridcolor=h.GRID)
fig.update_layout(height=420, template="plotly_white",
                  title="The dose–response the trial is trying to learn",
                  legend={"orientation": "h", "y": 1.12, "x": 0.0},
                  margin={"l": 60, "r": 30, "t": 100, "b": 50})
fig

## 7. The estimand is not the per-protocol curve

The curves above are what happens if every tablet is taken. Adherence is a
*consequence* of the assignment — it is lower at the higher dose, because the higher
dose has more side effects — so the quantity a randomized comparison licenses is the
**intent-to-treat** contrast: assigned dose against control, averaged over how much
was actually taken. Because the harm term is cubic in the dose taken, the gap between
the two is much wider for harm than for benefit. Notebook 2 makes that a formal
statement about a graph; here it is just arithmetic.

In [ ]:
rows = []
for arm in h.ARMS[1:]:
    dose = h.DOSE[arm]
    for stratum in h.STRATA:
        rows.append({"arm": h.ARM_LABEL[arm], "stratum": h.STRATUM_LABEL[stratum],
                     "per_protocol": h.per_protocol_contrast(dose, stratum),
                     "intent_to_treat": h.intent_to_treat_contrast(dose, stratum)})
    rows.append({"arm": h.ARM_LABEL[arm], "stratum": "population",
                 "per_protocol": h.per_protocol_contrast(dose),
                 "intent_to_treat": h.intent_to_treat_contrast(dose)})
estimands = pd.DataFrame(rows)
print("contrast against the standard of care at week 12 (mmHg; negative is better)")
print(estimands.pivot(index="stratum", columns="arm", values="intent_to_treat").round(2).to_string())
print("\nmean adherence by arm:")
print(units.groupby("arm")["adherence"].mean().round(3).to_string())

In [ ]:
fig = h.figure("Intent to treat is not per protocol — and the gap is worst where the harm is",
               "", f"contrast at week 12 ({h.OUTCOME_UNIT})", height=400, barmode="group")
subset = estimands[estimands["stratum"] != "population"]
labels = [f"{r.arm}<br>{r.stratum}" for r in subset.itertuples()]
fig.add_trace(go.Bar(x=labels, y=subset["per_protocol"], name="per protocol",
                     marker_color="rgba(90,100,115,0.55)"))
fig.add_trace(go.Bar(x=labels, y=subset["intent_to_treat"], name="intent to treat",
                     marker_color="#2f7fd1"))
fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.45)"})
fig

## 8. Enrollment, accrual and retention

Staggered entry is what makes an interim look a real event. At calendar week 14 the
trial has 400 randomized units but only half of them have finished the four weekly
readings the safety endpoint averages — and that share *is* the information fraction
the sequential design in notebook 5 runs on.

In [ ]:
weeks = list(range(6, h.TRIAL_WEEKS + 1))
enrolled = [int((units["enrolled_week"] <= w).sum()) for w in weeks]
safety = h.accrual(trial, weeks, endpoint="safety")
primary = h.accrual(trial, weeks, endpoint="primary")
fig = h.figure("Accrual: randomized, then complete on each endpoint",
               "calendar week", "units", height=400)
fig.add_trace(go.Scatter(x=weeks, y=enrolled, mode="lines", name="randomized",
                         line={"color": "#5b6472", "width": 2.5}))
fig.add_trace(go.Scatter(x=safety["calendar_week"], y=safety["units"], mode="lines",
                         name="safety window complete (weeks 5–8)",
                         line={"color": "#d1483f", "width": 2.5}))
fig.add_trace(go.Scatter(x=primary["calendar_week"], y=primary["units"], mode="lines",
                         name="primary window complete (weeks 9–12)",
                         line={"color": "#2f7fd1", "width": 2.5}))
fig.add_hline(y=trial.n_units, line={"color": "rgba(0,0,0,0.3)", "dash": "dot"})
fig

In [ ]:
completion = trial.completion()
fig = h.figure("Retention: share of units still on study", "weeks since randomization",
               "share still contributing readings", height=380)
for arm in h.ARMS:
    arm_units = set(units[units["arm"] == arm]["unit"])
    rows = trial.visits[trial.visits["unit"].isin(arm_units)]
    share = rows.groupby("week")["unit"].nunique() / len(arm_units)
    fig.add_trace(go.Scatter(x=share.index, y=share.to_numpy(), mode="lines",
                             line={"color": h.ARM_COLOR[arm], "width": 2.4}, name=h.ARM_LABEL[arm]))
fig.update_yaxes(range=[0.6, 1.02])
fig
print(f"overall retention at week {h.FOLLOW_UP_WEEKS}: {completion[h.FOLLOW_UP_WEEKS]:.1%}")

Retention is worse in the arm whose pressure is rising, because dropout in this world
depends on how a unit is doing. That is informative missingness, and notebook 2 says
what it costs.

## 9. Adherence and the noise the weekly schedule buys

Weekly measurement is not free. What it buys is precision: averaging the four readings
in a window rather than reading one visit cuts the variance of a unit's endpoint
almost in half, because the week-to-week noise is autocorrelated but far from
perfectly so. In notebook 5 that factor is the difference between a safety boundary
that can fire inside an age stratum and one that cannot.

In [ ]:
window = trial.visits[(trial.visits["week"] >= h.WINDOW["safety"][0])
                      & (trial.visits["week"] <= h.WINDOW["safety"][1])]
single = window[window["week"] == h.WINDOW["safety"][1]]
averaged = window.groupby("unit")["change"].mean()
resid_single = single.groupby("arm")["change"].std().mean()
resid_window = window.groupby("unit").agg(arm=("arm", "first"), change=("change", "mean")) \
    .groupby("arm")["change"].std().mean()
print(f"sd of a unit's endpoint, single visit  : {resid_single:.2f} {h.OUTCOME_UNIT}")
print(f"sd of a unit's endpoint, 4-week average: {resid_window:.2f} {h.OUTCOME_UNIT}")
print(f"variance ratio: {(resid_window / resid_single) ** 2:.3f}  "
      f"-> the same precision from {(resid_window / resid_single) ** 2:.0%} of the units")

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Adherence falls with dose", "One visit vs the four-week average"))
for arm in h.ARMS[1:]:
    fig.add_trace(go.Violin(y=units[units["arm"] == arm]["adherence"], name=h.ARM_LABEL[arm],
                            line_color=h.ARM_COLOR[arm], box_visible=True, meanline_visible=True,
                            showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=single["change"], name="single visit (week 8)", nbinsx=40,
                           marker_color="rgba(90,100,115,0.55)"), row=1, col=2)
fig.add_trace(go.Histogram(x=averaged, name="mean of weeks 5–8", nbinsx=40,
                           marker_color="rgba(47,127,209,0.65)"), row=1, col=2)
fig.update_yaxes(title_text="share of tablets taken", row=1, col=1, gridcolor=h.GRID)
fig.update_xaxes(title_text=f"change from baseline ({h.OUTCOME_UNIT})", row=1, col=2, gridcolor=h.GRID)
fig.update_layout(height=400, template="plotly_white", barmode="overlay",
                  legend={"orientation": "h", "y": 1.12, "x": 0.5},
                  margin={"l": 60, "r": 30, "t": 100, "b": 50})
fig.update_traces(opacity=0.72, selector={"type": "histogram"})
fig

## 10. Handing the trial to axiom

`data.Panel` is a frame plus a `RoleMap` that says which column is the unit, which is
time, which is the outcome, and which of the rest are treatments and covariates. It
sorts by (unit, time) on construction, which is the layout every downstream function
assumes. Notebook 4 fits a response surface on exactly this object.

In [ ]:
frame = trial.visits.rename(columns={"change": "sbp_change"})[
    ["unit", "week", "sbp_change", "dose", "baseline_sbp", "adherence"]
].copy()
panel = Panel(frame, RoleMap(unit="unit", time="week", outcome=("sbp_change", sbp),
                             treatments={"dose": drug},
                             covariates={"baseline_sbp": baseline, "adherence": adherence}))
print("panel:", panel.frame.shape, "| units:", panel.frame["unit"].nunique(),
      "| periods:", panel.frame["week"].nunique())
print("roles:", panel.roles.unit, panel.roles.time, panel.roles.outcome[0],
      list(panel.roles.treatments), list(panel.roles.covariates))
complete = panel.completeness()
print(f"\ncompleteness: {complete.n_units} units x {complete.n_periods} periods, "
      f"{complete.n_rows:,} rows, balanced={complete.balanced}, "
      f"{complete.missing_cells} missing cells ({complete.missing_cells / (complete.n_units * complete.n_periods):.1%})")
print("the missing cells are dropouts, not gaps in the middle:",
      f"{len(complete.gaps)} units end early")
print("\nunit of the outcome:", panel.roles.outcome[1].unit,
      "| unit of the treatment:", panel.roles.treatments["dose"].unit)
panel.frame.head()

## What this notebook established

- Randomization balanced baseline pressure **within** stratum; the between-stratum gap
  is ten times larger, which is why stratum enters every model as a variance term, not
  as a confounder.
- Pooled over the trial population the 40 mg arm looks like a modest, unremarkable
  benefit. Split by the stratum the protocol already randomized on, it is the best arm
  in the two younger bands and is **raising** pressure in the oldest one. A monitoring
  plan that only watches arms cannot see this.
- The quantity randomization licenses is the intent-to-treat contrast, and it is not
  the per-protocol curve — least of all for the harm, which is cubic in the dose
  actually taken.
- Staggered entry means information accrues over calendar time. Half the trial has a
  complete safety window by calendar week 14; that share is the information fraction
  every boundary in notebook 5 is indexed on.
- Weekly measurement buys a 4-week average whose variance is roughly half a single
  visit's. That factor is what makes stratum-level safety monitoring possible at all.